# Decision Trees: Soccer's Decision Flowchart

**Chapter 5: Advanced Classification Methods**

## Learning Objectives

- Understand how decision trees work
- Build a decision tree for pass completion prediction
- Visualize and interpret tree structure
- Understand tree-based decision rules
- Learn the advantages and disadvantages of decision trees

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsbombpy import sb
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## The Intuition: Learning 'If-Then' Rules

A **Decision Tree** is like playing "Twenty Questions" with your data.

**Example for shot prediction:**
1. Is shot distance < 18 meters? → If YES, go to question 2; if NO, predict "No Goal"
2. Is shot angle > 20 degrees? → If YES, go to question 3; if NO, predict "No Goal"
3. Was it a first-time shot? → If YES, predict "Goal"; if NO, predict "No Goal"

This creates a **transparent, human-readable model**.

## Load Data: Pass Completion Prediction

We'll predict whether a pass will be completed based on simple features.

In [ ]:
# Load match data
matches = sb.matches(competition_id=16, season_id=4)  # Champions League
events = sb.events(match_id=22912)  # Liverpool vs Tottenham Final

# Filter for passes
passes = events[events['type'] == 'Pass'].copy()

print(f"Total passes: {len(passes)}")
print(f"\nSample pass data:")
print(passes[['team', 'player', 'location', 'pass_outcome']].head())

## Feature Engineering

Create features that capture pass characteristics.

In [ ]:
# Extract pass features
passes['start_x'] = passes['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else None)
passes['start_y'] = passes['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else None)

# Calculate pass length
def calculate_pass_length(row):
    if pd.isna(row.get('pass_end_location')):
        return None
    start = row['location']
    end = row['pass_end_location']
    if isinstance(start, list) and isinstance(end, list):
        return np.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2)
    return None

passes['pass_length'] = passes.apply(calculate_pass_length, axis=1)

# Create target variable: 1 if completed, 0 if not
passes['completed'] = passes['pass_outcome'].isna().astype(int)

print(f"\nPass completion rate: {passes['completed'].mean():.2%}")
print(f"\nFeature summary:")
print(passes[['start_x', 'start_y', 'pass_length', 'completed']].describe())

## Prepare Data for Training

In [ ]:
# Select features and target
features = ['start_x', 'start_y', 'pass_length']
target = 'completed'

# Remove rows with missing values
X = passes[features].dropna()
y = passes.loc[X.index, target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} passes")
print(f"Test set: {len(X_test)} passes")
print(f"\nClass distribution in training set:")
print(y_train.value_counts(normalize=True))

## Train Decision Tree

We'll start with a simple tree with `max_depth=3` to keep it interpretable.

In [ ]:
# Train decision tree
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf.fit(X_train, y_train)

# Make predictions
y_pred = tree_clf.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Incomplete', 'Complete']))

## Visualize the Decision Tree

This is where decision trees shine - you can see exactly how they make decisions!

In [ ]:
# Visualize the tree
plt.figure(figsize=(20, 10))
plot_tree(tree_clf, 
          feature_names=features, 
          class_names=['Incomplete', 'Complete'],
          filled=True,
          fontsize=10,
          rounded=True)
plt.title('Decision Tree for Pass Completion Prediction', fontsize=16)
plt.tight_layout()
plt.show()

print("\nTree structure:")
print(f"Number of nodes: {tree_clf.tree_.node_count}")
print(f"Number of leaves: {tree_clf.tree_.n_leaves}")
print(f"Max depth: {tree_clf.tree_.max_depth}")

## Interpreting the Tree

Let's trace through a decision path:

**Example Rule:**
- If `pass_length ≤ 29.85` meters (short pass)
  - AND `start_x > 110.2` (in attacking third)
  - THEN predict **Complete** with high confidence (93%)

**Soccer Insight:** Short passes in the final third are highly likely to be completed. This aligns with tactical wisdom about maintaining possession in dangerous areas.

## Feature Importance

Which features matter most for the tree's decisions?

In [ ]:
# Get feature importances
importances = tree_clf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Feature Importances:")
print(feature_importance_df.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance in Decision Tree')
plt.tight_layout()
plt.show()

## Advantages of Decision Trees

| Advantage | Description |
|-----------|-------------|
| **Transparency** | White box model - you can explain every prediction |
| **No Feature Scaling** | Works with raw values, no normalization needed |
| **Handles Non-Linear Relationships** | Can model complex interactions |
| **Easy to Visualize** | Can literally see the decision logic |
| **Works with Mixed Data** | Handles both numerical and categorical features |

## Disadvantages of Decision Trees

| Disadvantage | Description |
|--------------|-------------|
| **High Variance** | Small changes in data → completely different tree |
| **Prone to Overfitting** | Deep trees memorize training data |
| **Limited Predictive Power** | Single tree often less accurate than ensembles |
| **Greedy Algorithm** | Makes locally optimal splits, may miss global optimum |

## Summary

In this notebook, we:

1. ✅ Understood how decision trees work (hierarchical if-then rules)
2. ✅ Built a tree for pass completion prediction
3. ✅ Visualized the tree structure
4. ✅ Interpreted decision rules in soccer context
5. ✅ Learned advantages and disadvantages

## Key Takeaways

- Decision trees are **highly interpretable**
- Great for **exploratory analysis** and presenting to non-technical stakeholders
- Single trees are **unstable** and prone to overfitting
- Need **tuning** (max_depth, min_samples_split) to prevent overfitting

## Next Steps

In the next notebook, we'll learn how to **tune decision trees** to prevent overfitting and improve performance!

## Practice Exercises

1. **Try Different Max Depths**: Train trees with max_depth from 1 to 10 and compare accuracy
2. **Add More Features**: Include pass height, body part, or pressure
3. **Different Outcome**: Predict shot outcomes instead of pass completion
4. **Visualize Decision Boundaries**: Plot the regions the tree creates in 2D feature space
5. **Extract Rules**: Write code to extract all decision rules as if-then statements